# 02 — Ligand Retrieval

Fetch compound–activity data from ChEMBL for each prioritized target and assign binary labels.

**Input:** `../data/processed/target_manifest.csv`
**Output:** `../data/processed/combined_bioactivity.csv`

In [ ]:
import sys, yaml
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
from src.acquisition import fetch_bioactivity_data, label_activity

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

manifest = pd.read_csv('../data/processed/target_manifest.csv')
print(f"Processing {len(manifest)} targets")
manifest[['gene_symbol', 'chembl_id', 'n_ligands']]

## Fetch bioactivity data from ChEMBL binding assays

In [ ]:
all_dfs = []
for _, row in manifest.iterrows():
    df = fetch_bioactivity_data(
        chembl_id=row['chembl_id'],
        activity_types=config['acquisition']['activity_types'],
        assay_type=config['acquisition']['assay_type'],
        max_compounds=config['acquisition']['max_compounds_per_target'],
    )
    if not df.empty:
        df['target_gene'] = row['gene_symbol']
        all_dfs.append(df)
        print(f"  {row['gene_symbol']:10s}: {len(df):>5d} compounds")

combined = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal: {len(combined)} compound–target pairs across {len(all_dfs)} targets")

## Binary activity labels

pChEMBL ≥ 6 (IC₅₀ ≤ 1 µM) → **active = 1**
pChEMBL < 5 (IC₅₀ > 10 µM) → **inactive = 0**
Gray zone (5–6) is dropped to keep class boundaries sharp.

In [ ]:
labeled = label_activity(
    combined,
    active_threshold=config['acquisition']['pchembl_cutoff_active'],
    inactive_threshold=config['acquisition']['pchembl_cutoff_inactive'],
)

n_active   = labeled['active'].sum()
n_inactive = (labeled['active'] == 0).sum()
print(f"Active: {n_active} | Inactive: {n_inactive} | Ratio: {n_inactive/n_active:.1f}:1")

labeled.to_csv('../data/processed/combined_bioactivity.csv', index=False)

## Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# pChEMBL distribution by target
for gene, grp in labeled.groupby('target_gene'):
    axes[0].hist(grp['pchembl_value'], bins=25, alpha=0.5, label=gene)
axes[0].axvline(config['acquisition']['pchembl_cutoff_active'],   color='green', ls='--', label='Active threshold')
axes[0].axvline(config['acquisition']['pchembl_cutoff_inactive'], color='red',   ls='--', label='Inactive threshold')
axes[0].set_xlabel('pChEMBL value')
axes[0].set_title('Activity distributions by target')
axes[0].legend(fontsize=7)

# Class balance per target
balance = labeled.groupby(['target_gene', 'active']).size().unstack(fill_value=0)
balance.columns = ['Inactive', 'Active']
balance.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71'])
axes[1].set_title('Class balance per target')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/processed/eda_activity.png', dpi=150)
plt.show()